# Lab 4 — Knowledge Graph + RAG Hybrid

**Goal:** Combine structured graph traversal with semantic vector search.  
Graph handles WHO/WHERE/WHAT relationships. RAG handles WHY/HOW/context.

**When to use which:**
- Graph: 'Who is the CEO of the company that acquired X?' (structured relation chain)
- RAG: 'What are the main challenges facing the company?' (semantic context)
- Hybrid: 'What did the CEO of X say about supply chain challenges?' (identify CEO via graph, find quotes via RAG)

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from dotenv import load_dotenv
load_dotenv(override=True)
from knowledge_graph import KnowledgeGraph, extract_triples
from openai import OpenAI
import numpy as np

client = OpenAI()
print('Ready ✓')

## 1. Build both a graph and a simple vector store from the same corpus

In [ ]:
DOCUMENTS = [
    """TechGiant Corp was founded by Maria Chen in 2010. She serves as CEO.
    The company is headquartered in Seattle and employs 50,000 people.""",
    
    """Maria Chen has spoken extensively about the importance of ethical AI.
    In a 2024 interview she said: 'We must ensure AI benefits everyone, not just the privileged few.'
    She has pushed for transparent AI development practices across the industry.""",
    
    """TechGiant acquired CloudBase Inc in 2022 for $3.2 billion.
    CloudBase was founded by James Park and is based in Austin, Texas.
    The acquisition expanded TechGiant's cloud computing capabilities significantly.""",
    
    """James Park, now a VP at TechGiant after the acquisition, has expressed concerns
    about rapid AI deployment. 'We need more guardrails,' he said at a 2023 conference.
    He advocates for slower, more deliberate AI rollout timelines.""",
]

# Build knowledge graph
kg = KnowledgeGraph()
for doc in DOCUMENTS:
    kg.add_text(doc)
print(f'Graph: {kg.stats()}')

# Build minimal vector store (embeddings)
def embed(text: str) -> list[float]:
    resp = client.embeddings.create(model='text-embedding-3-small', input=text)
    return resp.data[0].embedding

doc_embeddings = [(doc, embed(doc)) for doc in DOCUMENTS]
print(f'Vector store: {len(doc_embeddings)} documents')

In [ ]:
def vector_search(query: str, top_k: int = 2) -> list[str]:
    q_emb = np.array(embed(query))
    scores = []
    for doc, emb in doc_embeddings:
        score = np.dot(q_emb, np.array(emb)) / (np.linalg.norm(q_emb) * np.linalg.norm(np.array(emb)))
        scores.append((score, doc))
    scores.sort(reverse=True)
    return [doc for _, doc in scores[:top_k]]

## 2. Hybrid query: use graph to find entity, then RAG for context

In [ ]:
def hybrid_answer(question: str) -> str:
    # Step 1: Get graph context (structured facts)
    graph_context = kg.graph.edges(data=True)
    graph_triples = '\n'.join(
        f'({s}) --[{d["predicate"]}]--> ({o})' 
        for s, o, d in graph_context
    )
    
    # Step 2: Get RAG context (semantic documents)
    rag_docs = vector_search(question, top_k=2)
    rag_context = '\n\n'.join(rag_docs)
    
    # Step 3: Combined LLM answer
    prompt = f"""Answer the question using both the knowledge graph (for structured facts)
and the document excerpts (for quotes and context).

Knowledge graph:
{graph_triples}

Document excerpts:
{rag_context}

Question: {question}

Answer concisely, citing which source (graph/documents) provided each piece of information."""
    
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=300,
    )
    return resp.choices[0].message.content


# Pure structural question — graph excels
q1 = 'Who founded the company that TechGiant acquired?'
print(f'Q: {q1}')
print(f'A: {hybrid_answer(q1)}')
print()

In [ ]:
# Hybrid question — needs graph + RAG
q2 = "What did TechGiant's CEO say about AI ethics?"
print(f'Q: {q2}')
print(f'A: {hybrid_answer(q2)}')
print()

# Another hybrid — identify person via graph, get opinion via RAG
q3 = "What is the view of the founder of the company TechGiant acquired on AI deployment?"
print(f'Q: {q3}')
print(f'A: {hybrid_answer(q3)}')

## Exercise
Build a `query_router(question)` function that:
1. Classifies the question as 'graph', 'rag', or 'hybrid' using an LLM
2. Routes to the appropriate handler
3. Returns the answer with a tag showing which path was used

Test with at least 6 questions and verify the routing makes sense.

In [ ]:
# Your code here
